# Colab 2: L&O-NAE-SAT Experiments
## Reproducing Figure 2 (bottom-right) + Table 1

**Experiments**:
- **Figure 2 bottom-right**: Error imbalance across positions for (N=20, P=280)
- **Table 1**: Vanilla vs adaptive accuracy for 5 (N,P) configurations

**Model**: 19M MDM with RoPE, max sequence length 512

**Runtime**: T4 or A100 GPU, ~8-16 hours total

---
## Cell 1: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

!pip install torch torchvision transformers einops tqdm matplotlib seaborn -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
import math
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

---
## Cell 2: Model Architecture — MDM Transformer with RoPE

**Key properties**:
- **Bidirectional attention** (no causal mask) — MDM must see all unmasked tokens
- **RoPE** positional embeddings (Appendix C.2.1: "19M MDM with RoPE")
- **No time embedding** (Section 2: time-embedding-free architecture)
- 19M parameters: ~8 layers, 384 hidden, 6 heads, ff_dim=1536

In [ ]:
class RoPE(nn.Module):
    """Rotary Position Embedding (Su et al., 2021)."""
    def __init__(self, dim, max_seq_len=512):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len
    
    def forward(self, x, seq_len):
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1, causal=False):
        super().__init__()
        self.causal = causal
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, hidden_dim),
            nn.Dropout(dropout)
        )
        self.attn_dropout = nn.Dropout(dropout)
    
    def forward(self, x, rope_cos=None, rope_sin=None):
        B, L, D = x.shape
        h = self.ln1(x)
        qkv = self.qkv(h).reshape(B, L, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, L, D_h)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        if rope_cos is not None:
            q, k = apply_rotary_pos_emb(q, k, rope_cos, rope_sin)
        
        scale = math.sqrt(self.head_dim)
        attn = torch.matmul(q, k.transpose(-2, -1)) / scale
        
        if self.causal:
            causal_mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
            attn = attn.masked_fill(causal_mask[None, None], float('-inf'))
        
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, L, D)
        out = self.out_proj(out)
        
        x = x + out
        x = x + self.ff(self.ln2(x))
        return x


class MDMTransformer(nn.Module):
    """
    Masked Diffusion Model transformer.
    - BIDIRECTIONAL attention (causal=False)
    - NO time embedding
    - RoPE or learned positional embeddings
    """
    def __init__(self, vocab_size, hidden_dim, num_layers, num_heads, ff_dim,
                 max_seq_len=512, dropout=0.1, pos_type='rope'):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        
        self.pos_type = pos_type
        if pos_type == 'rope':
            self.rope = RoPE(hidden_dim // num_heads, max_seq_len)
            self.pos_emb = None
        elif pos_type == 'learned':
            self.pos_emb = nn.Embedding(max_seq_len, hidden_dim)
            self.rope = None
        
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads, ff_dim, dropout, causal=False)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x):
        B, L = x.shape
        h = self.embedding(x)
        
        if self.pos_type == 'learned' and self.pos_emb is not None:
            h = h + self.pos_emb(torch.arange(L, device=x.device))
        
        rope_cos, rope_sin = None, None
        if self.pos_type == 'rope' and self.rope is not None:
            rope_cos, rope_sin = self.rope(h, L)
        
        for block in self.blocks:
            h = block(h, rope_cos, rope_sin)
        
        h = self.ln_final(h)
        return self.output_head(h)


# Model configuration for ~19M params
# GPT-2 scaling: hidden=384, layers=8, heads=6, ff=1536
MODEL_CONFIG = {
    'hidden_dim': 384,
    'num_layers': 8,
    'num_heads': 6,
    'ff_dim': 1536,
}

# Vocabulary for L&O-NAE-SAT:
#   Data values: {0, 1} (observations), {1, 2} (latents), {2} (padding)
#   Distinct data values: {0, 1, 2}
#   Mask token: 3 (must be distinct from all data values)
MASK_TOKEN_ID = 3
VOCAB_SIZE = 4  # tokens 0, 1, 2 (data) + 3 (mask)

# Build model
model = MDMTransformer(
    vocab_size=VOCAB_SIZE,
    max_seq_len=512,
    pos_type='rope',
    **MODEL_CONFIG
)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params / 1e6:.1f}M")

---
## Cell 3: MDM Training Loop

**Loss** (Proposition E.1, Zheng et al. 2024):
1. Sample x0 from data
2. Sample n ~ Uniform(1, L) masks per sequence
3. Randomly mask n positions → x_masked
4. Loss = (1/n) × cross-entropy at masked positions

**No time embedding**: model takes only token IDs.

In [ ]:
def mdm_train_step(model, optimizer, x0, mask_token_id, device, pad_start=None):
    """
    Single MDM training step with proper 1/n loss weighting.
    
    Args:
        x0: (B, L) clean sequences
        mask_token_id: ID for mask token
        pad_start: if set, only mask positions < pad_start (don't mask padding)
    """
    model.train()
    B, L = x0.shape
    
    # Determine maskable positions
    if pad_start is not None:
        maskable_len = pad_start
    else:
        maskable_len = L
    
    # Sample n ~ Uniform(1, maskable_len) for each sequence
    n = torch.randint(1, maskable_len + 1, (B,), device=device)
    
    # Create masks using argsort trick for efficient random selection
    noise = torch.rand(B, maskable_len, device=device)
    sorted_indices = noise.argsort(dim=-1)
    
    mask = torch.zeros(B, L, dtype=torch.bool, device=device)
    for b in range(B):
        mask[b, sorted_indices[b, :n[b]]] = True
    
    # Create masked input
    x_masked = x0.clone()
    x_masked[mask] = mask_token_id
    
    # Forward pass
    logits = model(x_masked)  # (B, L, vocab_size)
    
    # Compute loss: per-sample cross-entropy at masked positions, weighted by 1/n
    total_loss = torch.tensor(0.0, device=device)
    for b in range(B):
        masked_logits = logits[b, mask[b]]    # (n_b, vocab_size)
        masked_targets = x0[b, mask[b]]       # (n_b,)
        if len(masked_targets) > 0:
            # Weight by 1/n as per the loss formulation
            sample_loss = F.cross_entropy(masked_logits, masked_targets, reduction='sum')
            total_loss = total_loss + sample_loss / n[b].float()
    
    total_loss = total_loss / B
    
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    return total_loss.item()


def train_mdm(model, train_data, mask_token_id, num_iterations, batch_size,
              lr=1e-3, weight_decay=0.1, device='cuda', save_path=None,
              pad_start=None):
    """
    Full MDM training loop.
    
    Optimizer: AdamW (beta1=0.9, beta2=0.95, wd=0.1)
    Scheduler: Cosine with min_lr = lr/10
    """
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr,
        betas=(0.9, 0.95), weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_iterations, eta_min=lr / 10
    )
    
    model = model.to(device)
    dataset_size = len(train_data)
    losses = []
    
    for step in tqdm(range(num_iterations), desc="Training MDM"):
        indices = np.random.randint(0, dataset_size, size=batch_size)
        x0 = torch.tensor(train_data[indices], dtype=torch.long, device=device)
        
        loss = mdm_train_step(model, optimizer, x0, mask_token_id, device, pad_start)
        scheduler.step()
        losses.append(loss)
        
        if (step + 1) % 200 == 0:
            avg_loss = np.mean(losses[-200:])
            print(f"  Step {step+1}/{num_iterations}, Loss: {avg_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")
    
    return losses

print("Training functions defined.")

---
## Cell 4: Inference Strategies

Three strategies (Section 4):
1. **Vanilla**: random unmasking order (baseline)
2. **Top probability**: unmask positions with highest max(p)
3. **Top probability margin**: unmask positions with largest |p(j1) - p(j2)|

**K** = (# masked) × (t - s) / t per step (linear schedule α_t = 1-t)

**Gumbel noise** coefficient = 0.5 for puzzles (Appendix D.2)

In [ ]:
@torch.no_grad()
def run_mdm_inference(model, seq_len, mask_token_id, vocab_size,
                      strategy='vanilla', num_steps=50,
                      gumbel_coeff=0.0, device='cuda',
                      fixed_tokens=None, fixed_mask=None):
    """
    MDM reverse-process inference.
    
    Args:
        strategy: 'vanilla', 'top_prob', or 'top_prob_margin'
        num_steps: 50 reverse sampling steps (Appendix D.2)
        gumbel_coeff: 0.5 for puzzles (Appendix D.2)
        fixed_tokens: (seq_len,) tensor of fixed values (for conditional generation)
        fixed_mask: (seq_len,) bool tensor (True = fixed/given, never modify)
    """
    model.eval()
    
    # Initialize fully masked (or conditionally masked)
    x = torch.full((1, seq_len), mask_token_id, dtype=torch.long, device=device)
    if fixed_tokens is not None and fixed_mask is not None:
        x[0, fixed_mask] = fixed_tokens[fixed_mask]
    
    # Linear noise schedule: alpha_t = 1 - t, reverse from t=1 to t=0
    timesteps = torch.linspace(1.0, 0.0, num_steps + 1)
    
    for step in range(num_steps):
        t = timesteps[step].item()
        s = timesteps[step + 1].item()
        
        # Identify currently masked positions (excluding fixed)
        is_masked = (x[0] == mask_token_id)
        if fixed_mask is not None:
            is_masked = is_masked & ~fixed_mask
        
        masked_positions = is_masked.nonzero(as_tuple=True)[0]
        num_masked = len(masked_positions)
        if num_masked == 0:
            break
        
        # K = num_masked * (t - s) / t  [linear schedule]
        unmask_frac = (t - s) / (t + 1e-10)
        K = max(1, round(num_masked * unmask_frac))
        K = min(K, num_masked)
        
        # Get model predictions
        logits = model(x)  # (1, seq_len, vocab_size)
        masked_logits = logits[0, masked_positions]  # (num_masked, vocab_size)
        
        # Exclude mask token from predictions
        masked_logits[:, mask_token_id] = float('-inf')
        probs = F.softmax(masked_logits, dim=-1)
        
        if strategy == 'vanilla':
            # Random selection of K positions to unmask
            perm = torch.randperm(num_masked, device=device)[:K]
            for idx in perm:
                token = torch.multinomial(probs[idx], 1).item()
                x[0, masked_positions[idx]] = token
        
        elif strategy == 'top_prob':
            # Certainty = max_j p(x^i = j)
            scores = probs.max(dim=-1).values
            if gumbel_coeff > 0:
                gumbel = -torch.log(-torch.log(torch.rand_like(scores) + 1e-8) + 1e-8)
                scores = scores + gumbel_coeff * gumbel
            _, top_k = torch.topk(scores, K)
            for idx in top_k:
                token = torch.multinomial(probs[idx], 1).item()
                x[0, masked_positions[idx]] = token
        
        elif strategy == 'top_prob_margin':
            # Certainty = |p(j1) - p(j2)| where j1, j2 are top-2 values
            top2_vals, _ = torch.topk(probs, k=min(2, probs.shape[-1]), dim=-1)
            if top2_vals.shape[-1] >= 2:
                scores = top2_vals[:, 0] - top2_vals[:, 1]
            else:
                scores = top2_vals[:, 0]
            if gumbel_coeff > 0:
                gumbel = -torch.log(-torch.log(torch.rand_like(scores) + 1e-8) + 1e-8)
                scores = scores + gumbel_coeff * gumbel
            _, top_k = torch.topk(scores, K)
            for idx in top_k:
                token = torch.multinomial(probs[idx], 1).item()
                x[0, masked_positions[idx]] = token
    
    # Fill any remaining masked positions
    remaining = (x[0] == mask_token_id)
    if fixed_mask is not None:
        remaining = remaining & ~fixed_mask
    if remaining.any():
        logits = model(x)
        for pos in remaining.nonzero(as_tuple=True)[0]:
            logits[0, pos, mask_token_id] = float('-inf')
            token = torch.multinomial(F.softmax(logits[0, pos], dim=-1), 1).item()
            x[0, pos] = token
    
    return x[0]

print("Inference functions defined.")

---
## Cell 5: Figure 2 Bottom-Right — Error Imbalance Experiment

**Setup (Appendix C.2.1)**:
- (N=20, P=280), m=2, padded to 512
- Train 19M MDM for 2×10³ iterations
- Train proxy (Bayes-optimal approximation) for 5×10⁴ iterations
- Mask ℓ=11 latent positions + ℓ×(P/N) observation positions
- Repeat 1000 times, measure squared error at each masked position

In [ ]:
# Load data
N, P = 20, 280
L = N + P  # 300
train_data = np.load(f'{BASE_DIR}/data/lo_nae_sat/N{N}_P{P}/train.npy')
test_data = np.load(f'{BASE_DIR}/data/lo_nae_sat/N{N}_P{P}/test.npy')
print(f"Train: {train_data.shape}, Test: {test_data.shape}")
print(f"Data value range: {train_data.min()} to {train_data.max()}")

# ---- Train the model (2000 iterations) ----
print("\n=== Training MDM (2000 iterations) ===")
model = MDMTransformer(
    vocab_size=VOCAB_SIZE, max_seq_len=512, pos_type='rope', **MODEL_CONFIG
).to(device)

losses = train_mdm(
    model, train_data, mask_token_id=MASK_TOKEN_ID,
    num_iterations=2000, batch_size=128,
    lr=0.001, device=device,
    save_path=f'{BASE_DIR}/models/lo_nae_sat_N20_P280_model.pt',
    pad_start=L  # Only mask positions 0..299, not padding
)

# ---- Train the proxy (50000 iterations) ----
print("\n=== Training Proxy MDM (50000 iterations) ===")
proxy_model = MDMTransformer(
    vocab_size=VOCAB_SIZE, max_seq_len=512, pos_type='rope', **MODEL_CONFIG
).to(device)

proxy_losses = train_mdm(
    proxy_model, train_data, mask_token_id=MASK_TOKEN_ID,
    num_iterations=50000, batch_size=128,
    lr=0.001, device=device,
    save_path=f'{BASE_DIR}/models/lo_nae_sat_N20_P280_proxy.pt',
    pad_start=L
)

In [ ]:
# ---- Measure Error Imbalance ----
# Appendix C.2.1: ell=11, 1000 trials
# "For each ell in [1, N-1], we randomly mask ell tokens in the latent positions
#  and ell * (P/N) tokens in the observed positions."

ell = 11
num_trials = 1000
errors = np.zeros(L)        # Accumulate squared error per position
error_counts = np.zeros(L)  # Count how many times each position was masked

model.eval()
proxy_model.eval()

for trial in tqdm(range(num_trials), desc="Measuring error imbalance"):
    idx = np.random.randint(len(test_data))
    x0 = test_data[idx]
    
    # Mask ell latent positions (from 0..N-1)
    lat_mask_idx = np.random.choice(N, size=ell, replace=False)
    # Mask ell * (P/N) observation positions (from N..N+P-1)
    obs_mask_count = int(ell * P / N)  # 11 * 280/20 = 154
    obs_mask_idx = np.random.choice(np.arange(N, N + P), size=obs_mask_count, replace=False)
    
    all_mask_idx = np.concatenate([lat_mask_idx, obs_mask_idx])
    
    x_masked = x0.copy()
    x_masked[all_mask_idx] = MASK_TOKEN_ID
    
    x_tensor = torch.tensor(x_masked, dtype=torch.long, device=device).unsqueeze(0)
    
    with torch.no_grad():
        logits_model = model(x_tensor)
        logits_proxy = proxy_model(x_tensor)
    
    for pos in all_mask_idx:
        if pos >= L:
            continue
        log_p_model = F.log_softmax(logits_model[0, pos], dim=-1)
        log_p_proxy = F.log_softmax(logits_proxy[0, pos], dim=-1)
        true_token = x0[pos]
        err = (log_p_model[true_token] - log_p_proxy[true_token]).item() ** 2
        errors[pos] += err
        error_counts[pos] += 1

# Average errors
error_counts = np.maximum(error_counts, 1)
avg_errors = errors / error_counts

np.save(f'{BASE_DIR}/results/fig2_bottom_right_errors.npy', avg_errors)

print(f"\nLatent positions (0-{N-1}) mean error:      {avg_errors[:N].mean():.6f}")
print(f"Observation positions ({N}-{N+P-1}) mean error: {avg_errors[N:L].mean():.6f}")
print("Expected: Latent errors HIGHER than observation errors.")

In [ ]:
# ---- Plot Figure 2 Bottom-Right ----
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['font.family'] = 'serif'

avg_errors = np.load(f'{BASE_DIR}/results/fig2_bottom_right_errors.npy')

fig, ax = plt.subplots(figsize=(8, 3.5))

positions = np.arange(L)
ax.bar(positions[:N], avg_errors[:N], color='#e74c3c', alpha=0.8, label='Latent positions', width=1.0)
ax.bar(positions[N:L], avg_errors[N:L], color='#3498db', alpha=0.6, label='Observation positions', width=1.0)

ax.set_xlabel('Position')
ax.set_ylabel('Prediction Error')
ax.set_title('Error Imbalance: L&O-NAE-SAT (N=20, P=280)')
ax.legend()
ax.set_xlim(-1, L + 1)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/figures/figure2_bottom_right.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{BASE_DIR}/figures/figure2_bottom_right.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure 2 bottom-right saved.")

---
## Cell 6: Table 1 — Vanilla vs Adaptive Accuracy

**5 configurations**: (N,P) = (25,275), (30,270), (40,260), (50,250), (100,200)

For each: train 19M MDM, evaluate vanilla and top_prob_margin inference.

**Expected results** (Table 1):
| (N,P) | Vanilla | Adaptive |
|-------|---------|----------|
| (25,275) | 78.06% | 93.76% |
| (30,270) | 75.70% | 93.54% |
| (40,260) | 74.60% | 92.21% |
| (50,250) | 67.94% | 90.01% |
| (100,200) | 62.84% | 88.91% |

In [ ]:
table1_results = {}

configs = [(25, 275), (30, 270), (40, 260), (50, 250), (100, 200)]

for N_cfg, P_cfg in configs:
    print(f"\n{'='*60}")
    print(f"Config (N={N_cfg}, P={P_cfg}), L={N_cfg+P_cfg}")
    print(f"{'='*60}")
    
    cfg_data = np.load(f'{BASE_DIR}/data/lo_nae_sat/N{N_cfg}_P{P_cfg}/train.npy')
    cfg_test = np.load(f'{BASE_DIR}/data/lo_nae_sat/N{N_cfg}_P{P_cfg}/test.npy')
    L_cfg = N_cfg + P_cfg
    
    # Build and train model
    cfg_model = MDMTransformer(
        vocab_size=VOCAB_SIZE, max_seq_len=512, pos_type='rope', **MODEL_CONFIG
    ).to(device)
    
    train_mdm(
        cfg_model, cfg_data, mask_token_id=MASK_TOKEN_ID,
        num_iterations=2000, batch_size=128,
        lr=0.001, device=device,
        save_path=f'{BASE_DIR}/models/lo_nae_sat_N{N_cfg}_P{P_cfg}.pt',
        pad_start=L_cfg
    )
    
    # Evaluate both strategies
    cfg_model.eval()
    num_eval = min(500, len(cfg_test))  # Evaluate on 500 samples
    
    for strategy_name in ['vanilla', 'top_prob_margin']:
        correct = 0
        total = 0
        gumbel = 0.5 if strategy_name == 'top_prob_margin' else 0.0
        
        for i in tqdm(range(num_eval), desc=f"Eval {strategy_name}"):
            x0 = cfg_test[i]
            
            # Pad positions are fixed; generate the rest from fully masked
            fixed_tokens = torch.tensor(x0, dtype=torch.long, device=device)
            fixed_mask = torch.zeros(512, dtype=torch.bool, device=device)
            fixed_mask[L_cfg:] = True  # Fix padding positions
            
            generated = run_mdm_inference(
                cfg_model, seq_len=512,
                mask_token_id=MASK_TOKEN_ID, vocab_size=VOCAB_SIZE,
                strategy=strategy_name, num_steps=50,
                gumbel_coeff=gumbel, device=device,
                fixed_tokens=fixed_tokens, fixed_mask=fixed_mask
            )
            
            # Check observation token accuracy (positions N..N+P-1)
            obs_pred = generated[N_cfg:N_cfg+P_cfg].cpu().numpy()
            obs_true = x0[N_cfg:N_cfg+P_cfg]
            obs_correct = (obs_pred == obs_true).sum()
            correct += obs_correct
            total += P_cfg
        
        accuracy = correct / total
        table1_results[(N_cfg, P_cfg, strategy_name)] = accuracy
        print(f"  {strategy_name}: {accuracy*100:.2f}%")

# Save results
with open(f'{BASE_DIR}/results/table1.json', 'w') as f:
    json.dump({str(k): v for k, v in table1_results.items()}, f, indent=2)

print("\nTable 1 results saved.")

In [ ]:
# ---- Display Table 1 ----
print("\n" + "="*60)
print("TABLE 1: L&O-NAE-SAT Vanilla vs Adaptive Accuracy")
print("="*60)
print(f"{'(N,P)':<12} {'Vanilla':>10} {'Adaptive':>10} {'Paper Van.':>12} {'Paper Adpt.':>12}")
print("-"*60)

paper_values = {
    (25, 275): (78.06, 93.76),
    (30, 270): (75.70, 93.54),
    (40, 260): (74.60, 92.21),
    (50, 250): (67.94, 90.01),
    (100, 200): (62.84, 88.91),
}

for N_cfg, P_cfg in configs:
    van = table1_results.get((N_cfg, P_cfg, 'vanilla'), 0) * 100
    adp = table1_results.get((N_cfg, P_cfg, 'top_prob_margin'), 0) * 100
    pv, pa = paper_values[(N_cfg, P_cfg)]
    print(f"({N_cfg},{P_cfg}){'':>4} {van:>9.2f}% {adp:>9.2f}% {pv:>11.2f}% {pa:>11.2f}%")

print("\nNaive guessing baseline: 75.00%")
print("Vanilla should decrease as N increases (harder problem).")
print("Adaptive should significantly outperform vanilla.")